# Chapter 24: Categorical Data, ANOVA, and Non-parametric Methods

**Level:** Applied  
**Objectives:** analyze contingency tables; compare independent groups with ANOVA, ranks, and permutation inference; interpret effect size.  
**Prerequisites:** Chapters 15 to 23.  
**Estimated study time:** 80 minutes.

All data are synthetic NRG warehouse observations generated with seed 20260829.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260829)
shifts = np.repeat(["Morning", "Evening", "Night"], 30)
picking_time = np.concatenate([rng.normal(18.5, 2.5, 30), rng.normal(20.5, 3.0, 30), rng.normal(23.0, 4.5, 30)])
late = np.concatenate([rng.binomial(1, 0.12, 30), rng.binomial(1, 0.20, 30), rng.binomial(1, 0.35, 30)])
print(f"Rows: {len(shifts)}")
print(f"Shift sizes: {[int(np.sum(shifts == s)) for s in ['Morning', 'Evening', 'Night']]}")

Rows: 90
Shift sizes: [30, 30, 30]


In [2]:
observed = np.array([[np.sum((shifts == s) & (late == 0)), np.sum((shifts == s) & (late == 1))] for s in ["Morning", "Evening", "Night"]])
expected = observed.sum(axis=1, keepdims=True) @ observed.sum(axis=0, keepdims=True) / observed.sum()
chi_square = np.sum((observed - expected) ** 2 / expected)
print(f"Observed late-status table:\n{observed}")
print(f"Chi-square: {chi_square:.3f}")

Observed late-status table:
[[26  4]
 [22  8]
 [17 13]]
Chi-square: 6.757


In [3]:
groups = [picking_time[shifts == s] for s in ["Morning", "Evening", "Night"]]
means = np.array([g.mean() for g in groups])
grand = picking_time.mean()
ss_between = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
ss_within = sum(np.sum((g - g.mean()) ** 2) for g in groups)
f_statistic = (ss_between / 2) / (ss_within / (len(picking_time) - 3))
eta_squared = ss_between / (ss_between + ss_within)
print(f"Group means: {[round(v, 2) for v in means]}")
print(f"F statistic: {f_statistic:.3f}")
print(f"Eta squared: {eta_squared:.3f}")

Group means: [18.88, 21.29, 22.28]
F statistic: 9.542
Eta squared: 0.180


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Morning", "Evening", "Night"], observed[:, 1] / observed.sum(axis=1), color=["#4c78a8", "#f2cf5b", "#e45756"])
axes[0].set(ylabel="Late proportion", title="Late shipments by shift", ylim=(0, 0.5))
axes[1].boxplot(groups, tick_labels=["Morning", "Evening", "Night"], showmeans=True)
axes[1].set(ylabel="Picking time (minutes)", title="Group distributions before testing")
fig.tight_layout()
plt.show()

<Figure size 1100x400 with 2 Axes>

In [5]:
order = np.argsort(picking_time, kind="stable")
ranks = np.empty(len(picking_time), dtype=float)
ranks[order] = np.arange(1, len(picking_time) + 1)
rank_sums = [ranks[shifts == s].sum() for s in ["Morning", "Evening", "Night"]]
h_statistic = 12 / (90 * 91) * sum(r ** 2 / 30 for r in rank_sums) - 3 * 91
permuted = []
for _ in range(999):
    labels = rng.permutation(shifts)
    split = [picking_time[labels == s] for s in ["Morning", "Evening", "Night"]]
    sb = sum(len(g) * (g.mean() - grand) ** 2 for g in split)
    sw = sum(np.sum((g - g.mean()) ** 2) for g in split)
    permuted.append((sb / 2) / (sw / 87))
p_value = (np.sum(np.array(permuted) >= f_statistic) + 1) / 1000
print(f"Kruskal-Wallis H: {h_statistic:.3f}")
print(f"Permutation p-value: {p_value:.4f}")

Kruskal-Wallis H: 18.668
Permutation p-value: 0.0010


In [6]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(permuted, bins=30, color="steelblue", edgecolor="white")
ax.axvline(f_statistic, color="firebrick", linewidth=2, label="Observed F")
ax.set(xlabel="Permuted F statistic", ylabel="Frequency", title="Null distribution from label permutations")
ax.legend()
fig.tight_layout()
plt.show()

<Figure size 700x400 with 1 Axes>

## Interpretation and limitations

The synthetic sample shows differences in late proportions and picking-time distributions across shifts. The statistics do not establish that shift assignment causes either outcome. Product mix, repeated workers, equipment, and time-specific disruptions may differ by shift. Eta squared describes sample association, while the permutation result depends on exchangeable labels.

## Practice

Calculate each cell's contribution to chi-square and compare planned mean differences between morning and night. Explain which result is most useful for a staffing decision. The cell below is intentionally blank.